### Notebook collecting commands/scripts which are run after fits/scans complete

In [5]:
# imports
# autoreload magic:
%load_ext autoreload
%autoreload 2
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from scipy.stats import norm
%matplotlib inline
import sys

from NNMFit.utilities import load_pickle
from NNMFit.utilities import ScanHandler

unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
sys.path.append(unblind_script_path)
from freefit_parameter_config import FreefitParamConfig

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


run scan analysis: calculate MC Histogram, Data Histogram, and Saturated LLH of the best fit

In [6]:
# new fits with updated data, muon template, bdts
fit_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits/"    

saturated_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step1_hese_flux_reunblind/saturated"
os.system(f"mkdir -p {saturated_path}")

config_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"
config_input_path = f"{config_path}/analysis_configs/data/hese/step1_hese_flux_reunblind/input_pars_only"
config_plot_path = f"{config_path}/analysis_configs/data/hese/step1_hese_flux_reunblind/plotting"
config_pse_path = f"{config_path}/analysis_configs/pse/hese/step1_hese_flux_reunblind"

os.system(f"mkdir -p {config_input_path}")
os.system(f"mkdir -p {config_plot_path}")
os.system(f"mkdir -p {config_pse_path}")

0

In [7]:
scans = {
    # "nbestfit50_all_param_2sigma_1D_10steps": 
    #     f"{fit_path}/nbestfit50_all_param_2sigma_1D_10steps",

    # without detector priors (`priors` key was not read by NNMFit):
    # "nbestfit50_all_param_2sigma_1D_10steps_PassingHESE": f"{fit_path}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE",
    # with the detector priors applied (fixed `prior` key)
    "nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_fixedprior":
        f"{fit_path}/nbestfit50_all_param_2sigma_1D_10steps_PassingHESE_fixedprior",
}

In [8]:
for name, scan_dir in scans.items():
    # run do_scan_analysis.py with scan_dir as arg:
    print(f"Processing scan directory: {scan_dir}")
    os.system(f"python {unblind_script_path}/do_scan_analysis.py {scan_dir} --sat_llh_save_dir {saturated_path}/{name}")

Processing scan directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE
Auto-Detecting the best fit from among the freefits in the scan path.
Initializing on server cobalt-14. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only


/mnt/ceph1-npx/user/tvaneede/software/NNMFit_installations/py3-v4.2.1_nnmfit-v0.5.0/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'combined_muongun_norm', 'conv_norm', 'delta_gamma', 'dom_eff',
       'fit_success', 'gamma_astro', 'ice_abs', 'ice_crystal', 'ice_holep0',
       'ice_holep1', 'ice_scat', 'llh', 'muongun_asymmetry', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


found the minimum llh value:  182.03479634514542
Found 50 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE
This is the index of the best fit: 3, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE/Freefit_03.pickle
found the minimum llh value:  182.03479634514542
Found 50 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE
This is the index of the best fit: 3, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE/Freefit_03.pickle
Data Histo

Collect the bestfit parameter values in a config file (for later pseudoexperiments etc.)

In [9]:
# find the best-fit for each scan and write a configuration file using the
# bestfit params as input params:
freefit_config_names = {}
min_llh_dict = {}
scan_llh = {}
for name, scan_path in scans.items():
    print(scan_path)
    freefit_config_hdl = FreefitParamConfig(scan_path=scan_path)
    print(freefit_config_hdl.get_name())
    freefit_config_hdl.write(custom_name=f"{config_input_path}/{name}.yaml")
    llh = freefit_config_hdl.get_min_llh_freefit(freefit_config_hdl.scan_hdl)
    min_llh_dict[freefit_config_hdl.get_name()] = llh
    scan_llh[name] = llh
    freefit_config_names[os.path.basename(os.path.dirname(scan_path))
                        ] = freefit_config_hdl.get_name()


/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE
Initializing on server cobalt-14. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only
found the minimum llh value:  182.03479634514542
Found 50 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE
This is the index of the best fit: 3, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step1_hese_flux_try2/data_fits//nbestfit50_all_param_2sigma_1D_10steps_PassingHESE/Freefit_03.pickle
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only/Input_params_data_fits_nbestfit50_all_param_2sig

/mnt/ceph1-npx/user/tvaneede/software/NNMFit_installations/py3-v4.2.1_nnmfit-v0.5.0/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'barr_h', 'barr_w', 'barr_y', 'barr_z',
       'combined_muongun_norm', 'conv_norm', 'delta_gamma', 'dom_eff',
       'fit_success', 'gamma_astro', 'ice_abs', 'ice_crystal', 'ice_holep0',
       'ice_holep1', 'ice_scat', 'llh', 'muongun_asymmetry', 'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


In [10]:
# For each scan, create a subfolder with full data configs (separate + combined detectors)
# and component-only variants (Astro / per-flavor Astro / Conv / Prompt / Muon / MuonConv / MuonPrompt).
import yaml
import copy

DETECTOR_CONFIGS_SEPARATE = [
    "IC86_pass2_SnowStorm_FTP_HESE_Cascades",
    "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades",
    "IC86_pass2_SnowStorm_FTP_HESE_Tracks",
]
DETECTOR_CONFIGS_COMBINED = ["IC86_pass2_SnowStorm_FTP_HESE_Combined"]


def write_data_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: data\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")


def write_pse_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: pseudoexp\n")
        f.write("pseudoexp_type: poisson\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")

for scan_name,scan_path in scans.items():
    input_yaml_path = f"{config_input_path}/{scan_name}.yaml"
    with open(input_yaml_path) as f:
        base_data = yaml.safe_load(f)
    params = base_data["input_params"]
    # 3-flavor parametrization (astro_SPL_3flavor) equivalent to the single astro_norm (1:1:1),
    # only used for the per-flavor plots
    params["total_astro_norm"] = 3*params["astro_norm"]
    params["gamma"] = params["gamma_astro"]
    params["a"] = 0.4444444444
    params["b"] = 0.0

    # all flux normalizations, every component switches off all but its own
    ALL_NORMS_OFF = {"astro_norm": 0, "total_astro_norm": 0, "conv_norm": 0, "prompt_norm": 0, "combined_muongun_norm": 0}

    # split the combined muon template into its conv/prompt parts:
    # mu = N*[(1+a)*T_conv + (1-a)*T_prompt], so the conv part is reproduced by
    # norm N(1+a)/2 at a=+1, and the prompt part by norm N(1-a)/2 at a=-1
    muon_norm = params["combined_muongun_norm"]
    muon_asym = params["muongun_asymmetry"]

    COMPONENT_OVERRIDES = {
        "Astro":      {**ALL_NORMS_OFF, "astro_norm": params["astro_norm"]},
        # per-flavor astro, with astro_SPL_3flavor_no_inel.yaml
        "Astro_3flavor": {**ALL_NORMS_OFF, "total_astro_norm": params["total_astro_norm"], "a": 0.4444444444, "b": 0.0},
        "Astro_NuE":     {**ALL_NORMS_OFF, "total_astro_norm": params["total_astro_norm"]/3, "a": 1.0, "b": 1.0},
        "Astro_NuMu":    {**ALL_NORMS_OFF, "total_astro_norm": params["total_astro_norm"]/3, "a": 1.0, "b": -1.0},
        "Astro_NuTau":   {**ALL_NORMS_OFF, "total_astro_norm": params["total_astro_norm"]/3, "a": 0.0, "b": 0.0},
        "Conv":       {**ALL_NORMS_OFF, "conv_norm": params["conv_norm"]},
        "Prompt":     {**ALL_NORMS_OFF, "prompt_norm": params["prompt_norm"]},
        "Muon":       {**ALL_NORMS_OFF, "combined_muongun_norm": muon_norm},
        "MuonConv":   {**ALL_NORMS_OFF, "combined_muongun_norm": muon_norm * (1 + muon_asym) / 2, "muongun_asymmetry": 1.0},
        "MuonPrompt": {**ALL_NORMS_OFF, "combined_muongun_norm": muon_norm * (1 - muon_asym) / 2, "muongun_asymmetry": -1.0},
    }

    # pse configs
    write_pse_config(params, DETECTOR_CONFIGS_SEPARATE, f"{config_pse_path}/{scan_name}.yaml")
    print(f"Written: {config_pse_path}/{scan_name}.yaml")

    # data configs for plotting
    scan_dir = f"{config_plot_path}/{scan_name}"
    os.makedirs(scan_dir, exist_ok=True)

    variants = [
        (scan_name,                    DETECTOR_CONFIGS_SEPARATE),
        (f"{scan_name}_combined",      DETECTOR_CONFIGS_COMBINED),
    ]

    for stem, det_configs in variants:
        write_data_config(params, det_configs, f"{scan_dir}/{stem}.yaml")
        print(f"Written: {scan_dir}/{stem}.yaml")

        for component, overrides in COMPONENT_OVERRIDES.items():
            write_data_config(params, det_configs, f"{scan_dir}/{stem}_{component}.yaml", overrides=overrides)
            print(f"Written: {scan_dir}/{stem}_{component}.yaml")

    # best fit with the crystal density scaling (ice_crystal) fixed by hand, separate detectors only
    CRYSTAL_VALUES = [0.9, 1.0, 1.1, 1.2]
    for crystal in CRYSTAL_VALUES:
        stem = f"{scan_name}_ice_crystal_{crystal}"
        write_data_config(params, DETECTOR_CONFIGS_SEPARATE, f"{scan_dir}/{stem}.yaml", overrides={"ice_crystal": crystal})
        print(f"Written: {scan_dir}/{stem}.yaml")

        for component, overrides in COMPONENT_OVERRIDES.items():
            write_data_config(params, DETECTOR_CONFIGS_SEPARATE, f"{scan_dir}/{stem}_{component}.yaml",
                              overrides={**overrides, "ice_crystal": crystal})
            print(f"Written: {scan_dir}/{stem}_{component}.yaml")